In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import random
import numpy as np
from typing import List, Tuple, Dict

class CNNGenome:
    """Represents a CNN architecture genome"""
    def __init__(self, genome=None):
        if genome is None:
            # Random initialization
            self.conv_layers = random.randint(1, 4)  # 1-4 conv layers
            self.filters = [random.choice([16, 32, 64, 128]) for _ in range(self.conv_layers)]
            self.kernel_sizes = [random.choice([3, 5]) for _ in range(self.conv_layers)]
            self.fc_layers = random.randint(1, 3)  # 1-3 fully connected layers
            self.fc_sizes = [random.choice([64, 128, 256, 512]) for _ in range(self.fc_layers)]
            self.dropout_rate = random.uniform(0.1, 0.5)
        else:
            # Copy from existing genome
            self.conv_layers = genome.conv_layers
            self.filters = genome.filters.copy()
            self.kernel_sizes = genome.kernel_sizes.copy()
            self.fc_layers = genome.fc_layers
            self.fc_sizes = genome.fc_sizes.copy()
            self.dropout_rate = genome.dropout_rate
    
    def mutate(self, mutation_rate=0.1):
        """Apply random mutations to the genome"""
        if random.random() < mutation_rate:
            # Mutate number of conv layers
            self.conv_layers = max(1, min(4, self.conv_layers + random.choice([-1, 1])))
            self.filters = self.filters[:self.conv_layers]
            self.kernel_sizes = self.kernel_sizes[:self.conv_layers]
            
            # Add new layers if needed
            while len(self.filters) < self.conv_layers:
                self.filters.append(random.choice([16, 32, 64, 128]))
                self.kernel_sizes.append(random.choice([3, 5]))
        
        # Mutate existing layers
        for i in range(len(self.filters)):
            if random.random() < mutation_rate:
                self.filters[i] = random.choice([16, 32, 64, 128])
            if random.random() < mutation_rate:
                self.kernel_sizes[i] = random.choice([3, 5])
        
        # Mutate FC layers
        if random.random() < mutation_rate:
            self.fc_layers = max(1, min(3, self.fc_layers + random.choice([-1, 1])))
            self.fc_sizes = self.fc_sizes[:self.fc_layers]
            while len(self.fc_sizes) < self.fc_layers:
                self.fc_sizes.append(random.choice([64, 128, 256, 512]))
        
        for i in range(len(self.fc_sizes)):
            if random.random() < mutation_rate:
                self.fc_sizes[i] = random.choice([64, 128, 256, 512])
        
        if random.random() < mutation_rate:
            self.dropout_rate = random.uniform(0.1, 0.5)

class DynamicCNN(nn.Module):
    """CNN that builds itself from a genome"""
    def __init__(self, genome: CNNGenome, input_channels=3, num_classes=10):
        super(DynamicCNN, self).__init__()
        self.genome = genome
        
        # Build convolutional layers
        conv_layers = []
        in_channels = input_channels
        
        for i in range(genome.conv_layers):
            conv_layers.extend([
                nn.Conv2d(in_channels, genome.filters[i], 
                         kernel_size=genome.kernel_sizes[i], padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2)
            ])
            in_channels = genome.filters[i]
        
        self.conv_layers = nn.Sequential(*conv_layers)
        
        # Calculate the size after conv layers (assuming 32x32 input)
        with torch.no_grad():
            x = torch.randn(1, input_channels, 32, 32)
            x = self.conv_layers(x)
            conv_output_size = x.view(1, -1).size(1)
        
        # Build fully connected layers
        fc_layers = []
        in_features = conv_output_size
        
        for i, fc_size in enumerate(genome.fc_sizes):
            fc_layers.extend([
                nn.Linear(in_features, fc_size),
                nn.ReLU(),
                nn.Dropout(genome.dropout_rate)
            ])
            in_features = fc_size
        
        # Final output layer
        fc_layers.append(nn.Linear(in_features, num_classes))
        self.fc_layers = nn.Sequential(*fc_layers)
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc_layers(x)
        return x

class GeneticAlgorithm:
    """Genetic Algorithm for CNN evolution"""
    def __init__(self, population_size=20, mutation_rate=0.1, elite_size=5):
        self.population_size = population_size
        self.mutation_rate = mutation_rate
        self.elite_size = elite_size
        self.population = [CNNGenome() for _ in range(population_size)]
        self.fitness_scores = []
    
    def evaluate_fitness(self, genome: CNNGenome, train_loader, val_loader, 
                        input_channels=3, num_classes=10, epochs=5):
        """Evaluate fitness of a genome by training and testing the CNN"""
        try:
            model = DynamicCNN(genome, input_channels, num_classes)
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            model.to(device)
            
            optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
            criterion = nn.CrossEntropyLoss()
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
            
            # Training with more epochs for CIFAR-10
            model.train()
            for epoch in range(epochs):
                train_loss = 0
                batches_processed = 0
                for batch_idx, (data, target) in enumerate(train_loader):
                    if batch_idx > 50:  # Process more batches for better evaluation
                        break
                    data, target = data.to(device), target.to(device)
                    optimizer.zero_grad()
                    output = model(data)
                    loss = criterion(output, target)
                    loss.backward()
                    optimizer.step()
                    train_loss += loss.item()
                    batches_processed += 1
                
                scheduler.step()
                avg_loss = train_loss / batches_processed
                print(f"    Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
            
            # Validation on CIFAR-10 test set
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for batch_idx, (data, target) in enumerate(val_loader):
                    if batch_idx > 30:  # Test on more validation batches
                        break
                    data, target = data.to(device), target.to(device)
                    outputs = model(data)
                    _, predicted = torch.max(outputs.data, 1)
                    total += target.size(0)
                    correct += (predicted == target).sum().item()
            
            accuracy = 100 * correct / total
            
            # Enhanced fitness function for CIFAR-10
            param_count = sum(p.numel() for p in model.parameters())
            
            # Reward accuracy, penalize excessive parameters
            fitness = accuracy - (param_count / 500000)  # Adjusted penalty for CIFAR-10
            
            # Bonus for achieving good accuracy milestones
            if accuracy > 70:
                fitness += 5
            if accuracy > 80:
                fitness += 10
            
            print(f"    Accuracy: {accuracy:.2f}%, Parameters: {param_count:,}, Fitness: {fitness:.2f}")
            
            return max(0, fitness)  # Ensure non-negative fitness
            
        except Exception as e:
            print(f"    Error evaluating genome: {e}")
            return 0  # Failed models get 0 fitness
    
    def select_parents(self) -> Tuple[CNNGenome, CNNGenome]:
        """Tournament selection"""
        def tournament():
            candidates = random.sample(list(zip(self.population, self.fitness_scores)), 3)
            return max(candidates, key=lambda x: x[1])[0]
        
        return tournament(), tournament()
    
    def crossover(self, parent1: CNNGenome, parent2: CNNGenome) -> CNNGenome:
        """Create offspring by combining parent genomes"""
        child = CNNGenome()
        
        # Randomly choose traits from parents
        child.conv_layers = random.choice([parent1.conv_layers, parent2.conv_layers])
        child.filters = (parent1.filters + parent2.filters)[:child.conv_layers]
        child.kernel_sizes = (parent1.kernel_sizes + parent2.kernel_sizes)[:child.conv_layers]
        
        child.fc_layers = random.choice([parent1.fc_layers, parent2.fc_layers])
        child.fc_sizes = (parent1.fc_sizes + parent2.fc_sizes)[:child.fc_layers]
        
        child.dropout_rate = random.choice([parent1.dropout_rate, parent2.dropout_rate])
        
        return child
    
    def evolve_generation(self, train_loader, val_loader, input_channels=3, num_classes=10):
        """Evolve one generation"""
        print(f"Evaluating {len(self.population)} individuals...")
        
        # Evaluate fitness for entire population
        self.fitness_scores = []
        for i, genome in enumerate(self.population):
            fitness = self.evaluate_fitness(genome, train_loader, val_loader, 
                                          input_channels, num_classes)
            self.fitness_scores.append(fitness)
            print(f"Individual {i+1}: Fitness = {fitness:.2f}")
        
        # Sort by fitness
        sorted_pop = sorted(zip(self.population, self.fitness_scores), 
                           key=lambda x: x[1], reverse=True)
        
        # Keep elite
        new_population = [genome for genome, _ in sorted_pop[:self.elite_size]]
        
        # Generate offspring
        while len(new_population) < self.population_size:
            parent1, parent2 = self.select_parents()
            child = self.crossover(parent1, parent2)
            child.mutate(self.mutation_rate)
            new_population.append(child)
        
        self.population = new_population
        best_fitness = sorted_pop[0][1]
        print(f"Best fitness this generation: {best_fitness:.2f}")
        
        return sorted_pop[0][0], best_fitness  # Return best genome and fitness

# CIFAR-10 Dataset Setup
def load_cifar10_data():
    """Load and prepare CIFAR-10 dataset"""
    import torchvision
    import torchvision.transforms as transforms
    
    # Data augmentation and normalization for training
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    # Normalization for validation
    transform_val = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    # Load CIFAR-10 dataset
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                          download=True, transform=transform_train)
    
    valset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform_val)
    
    # Create data loaders
    train_loader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
    val_loader = DataLoader(valset, batch_size=100, shuffle=False, num_workers=2)
    
    return train_loader, val_loader

def get_cifar10_classes():
    """Return CIFAR-10 class names"""
    return ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

def main():
    """Main evolution loop using CIFAR-10 dataset"""
    print("Starting CNN Evolution with Genetic Algorithm on CIFAR-10")
    print("Loading CIFAR-10 dataset...")
    
    # Load CIFAR-10 data
    train_loader, val_loader = load_cifar10_data()
    classes = get_cifar10_classes()
    print(f"CIFAR-10 classes: {classes}")
    print(f"Training samples: {len(train_loader.dataset)}")
    print(f"Validation samples: {len(val_loader.dataset)}")
    
    # Initialize genetic algorithm
    ga = GeneticAlgorithm(population_size=8, mutation_rate=0.15, elite_size=2)
    
    # Evolution loop
    generations = 10
    best_genome = None
    best_fitness = 0
    
    fitness_history = []
    
    for generation in range(generations):
        print(f"\n{'='*50}")
        print(f"Generation {generation + 1}/{generations}")
        print(f"{'='*50}")
        
        genome, fitness = ga.evolve_generation(train_loader, val_loader, 
                                             input_channels=3, num_classes=10)
        
        fitness_history.append(fitness)
        
        if fitness > best_fitness:
            best_fitness = fitness
            best_genome = genome
            print(f"🎉 New best fitness: {best_fitness:.2f}")
    
    print(f"\n{'='*50}")
    print(f"EVOLUTION COMPLETE")
    print(f"{'='*50}")
    print(f"Best fitness achieved: {best_fitness:.2f}")
    print(f"\nBest architecture found:")
    print(f"  Conv layers: {best_genome.conv_layers}")
    print(f"  Filters per layer: {best_genome.filters}")
    print(f"  Kernel sizes: {best_genome.kernel_sizes}")
    print(f"  FC layers: {best_genome.fc_layers}")
    print(f"  FC layer sizes: {best_genome.fc_sizes}")
    print(f"  Dropout rate: {best_genome.dropout_rate:.3f}")
    
    # Final evaluation of best model
    print(f"\nFinal evaluation of best model...")
    best_model = DynamicCNN(best_genome, input_channels=3, num_classes=10)
    param_count = sum(p.numel() for p in best_model.parameters())
    print(f"Total parameters: {param_count:,}")
    
    # Show fitness progression
    print(f"\nFitness progression over generations:")
    for i, fit in enumerate(fitness_history):
        print(f"  Generation {i+1}: {fit:.2f}")
    
    return best_model, best_genome

# Additional utility function for testing the best model
def test_best_model(model, test_loader, classes):
    """Test the evolved model on CIFAR-10 test set"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    correct = 0
    total = 0
    class_correct = list(0. for i in range(10))
    class_total = list(0. for i in range(10))
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
            # Per-class accuracy
            c = (predicted == target).squeeze()
            for i in range(target.size(0)):
                label = target[i]
                class_correct[label] += c[i].item()
                class_total[label] += 1
    
    print(f'\nOverall Test Accuracy: {100 * correct / total:.2f}%')
    print('\nPer-class accuracy:')
    for i in range(10):
        if class_total[i] > 0:
            print(f'{classes[i]}: {100 * class_correct[i] / class_total[i]:.2f}%')

if __name__ == "__main__":
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)
    
    best_model, best_genome = main()
    
    # Optional: Test the best evolved model
    print("\nTesting evolved model on full CIFAR-10 test set...")
    train_loader, test_loader = load_cifar10_data()
    classes = get_cifar10_classes()
    test_best_model(best_model, test_loader, classes)

Starting CNN Evolution with Genetic Algorithm on CIFAR-10
Loading CIFAR-10 dataset...


100%|██████████| 170M/170M [00:03<00:00, 55.0MB/s] 


CIFAR-10 classes: ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
Training samples: 50000
Validation samples: 10000

Generation 1/10
Evaluating 8 individuals...
    Epoch 1/5, Loss: 2.0108
    Epoch 2/5, Loss: 1.7710
    Epoch 3/5, Loss: 1.6867
    Epoch 4/5, Loss: 1.6047
    Epoch 5/5, Loss: 1.5742
    Accuracy: 46.74%, Parameters: 463,434, Fitness: 45.82
Individual 1: Fitness = 45.82
    Epoch 1/5, Loss: 2.0598
    Epoch 2/5, Loss: 1.8400
    Epoch 3/5, Loss: 1.7643
    Epoch 4/5, Loss: 1.6744
    Epoch 5/5, Loss: 1.6640
    Accuracy: 44.65%, Parameters: 232,330, Fitness: 44.18
Individual 2: Fitness = 44.18
    Epoch 1/5, Loss: 2.1420
    Epoch 2/5, Loss: 1.9476
    Epoch 3/5, Loss: 1.8402
    Epoch 4/5, Loss: 1.7699
    Epoch 5/5, Loss: 1.7343
    Accuracy: 41.16%, Parameters: 424,618, Fitness: 40.31
Individual 3: Fitness = 40.31
    Epoch 1/5, Loss: 2.0164
    Epoch 2/5, Loss: 1.7344
    Epoch 3/5, Loss: 1.6239
    Epoch 4/5, Loss: 1.5438
    Epoch 